# 01 Explore ASL alphabet dataset
Run on Colab with a T4 GPU. Replace `<you>` in the clone URL with your GitHub user.


In [ ]:
# 1) Mount Google Drive (a popup asks permission)
from google.colab import drive
drive.mount("/content/drive")

# 2) Paths
DRIVE = "/content/drive/MyDrive/sign-language"

# 3) Code + packages
!git clone https://github.com/<you>/sign-language-translator.git /content/repo 2>/dev/null || git -C /content/repo pull
!pip install -q mediapipe==0.10.14 "numpy<2" scikit-learn seaborn tqdm
%cd /content/repo
import sys
sys.path.insert(0, "/content/repo")



Download Kaggle `grassknoted/asl-alphabet` and inspect class counts. Skip `nothing` later during preprocess.


In [ ]:
!mkdir -p ~/.kaggle
!cp {DRIVE}/kaggle/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d grassknoted/asl-alphabet -p /content/raw
!unzip -q /content/raw/asl-alphabet.zip -d /content/raw/asl
!ls /content/raw/asl



In [ ]:
from pathlib import Path
from collections import Counter
import random
import cv2
import matplotlib.pyplot as plt

from src.config import ALPHABET_LABELS

SRC = Path("/content/raw/asl/asl_alphabet_train/asl_alphabet_train")
print("exists", SRC.exists(), "children", [p.name for p in SRC.iterdir()][:10] if SRC.exists() else None)

counts = {}
for p in sorted(SRC.iterdir()) if SRC.exists() else []:
    counts[p.name] = len(list(p.glob("*.jpg")))
print(counts)

# show one sample per alphabet label if present
fig, axes = plt.subplots(4, 7, figsize=(14, 8))
axes = axes.ravel()
for i, lab in enumerate(ALPHABET_LABELS):
    files = list((SRC / lab).glob("*.jpg"))
    if not files:
        axes[i].set_title(lab + " missing")
        axes[i].axis("off")
        continue
    img = cv2.cvtColor(cv2.imread(str(files[0])), cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(lab)
    axes[i].axis("off")
plt.tight_layout()
plt.show()
print("Note: J and Z are motion letters. Static image models will be weaker on them.")

